# 👥 Employee Salary & Attrition Predictor
Predicts **Monthly Income** (regression) and **Attrition risk** (classification) from HR data. Trains multiple algorithms, auto-picks the best one for each task, and serves predictions through a Gradio UI.

**Run all cells top to bottom** (Runtime → Run all). You'll be asked to upload your CSV in Step 1.

## 1. Setup

In [ ]:
!pip install -q gradio xgboost

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from xgboost import XGBRegressor, XGBClassifier
import gradio as gr
import joblib


## 2. Load Data
Upload your CSV (e.g. `ibm_hr.csv`) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()
csv_name = list(uploaded.keys())[0]
df = pd.read_csv(csv_name)
print(df.shape)
df.head()


## 3. Quick Clean
Drops duplicates and fills any missing values. (Your current dataset already has none, but this makes the notebook reusable with messier data too.)

In [ ]:
df = df.drop_duplicates().reset_index(drop=True)

for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

print('Missing values left:', df.isnull().sum().sum())
print('Rows after dedup:', len(df))


## 4. Features & Targets
- **Regression target:** `MonthlyIncome`
- **Classification target:** `Attrition` (Yes/No → 1/0)
- Both targets are dropped from the shared feature set so neither leaks into the other.

In [ ]:
TARGET_REG = 'MonthlyIncome'
TARGET_CLF = 'Attrition'

X = df.drop(columns=[TARGET_REG, TARGET_CLF])
y_reg = df[TARGET_REG]
y_clf = df[TARGET_CLF].map({'Yes': 1, 'No': 0}) if df[TARGET_CLF].dtype == 'object' else df[TARGET_CLF]

numeric_cols = X.select_dtypes(include=np.number).columns.tolist()
categorical_cols = X.select_dtypes(exclude=np.number).columns.tolist()
print('Numeric features:', len(numeric_cols))
print('Categorical features:', len(categorical_cols))


In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

X_train, X_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size=0.2, random_state=42, stratify=y_clf
)

X_train_proc = preprocessor.fit_transform(X_train)
X_test_proc = preprocessor.transform(X_test)
print('Train shape:', X_train_proc.shape, '| Test shape:', X_test_proc.shape)


## 5. Train & Compare Regression Models (Monthly Income)

In [ ]:
reg_models = {
    'Linear Regression': LinearRegression(),
    'Decision Tree': DecisionTreeRegressor(random_state=42),
    'Random Forest': RandomForestRegressor(random_state=42, n_estimators=200),
    'XGBoost': XGBRegressor(random_state=42, n_estimators=200, verbosity=0),
}

reg_results = []
trained_reg = {}
for name, model in reg_models.items():
    model.fit(X_train_proc, y_reg_train)
    preds = model.predict(X_test_proc)
    reg_results.append({
        'Model': name,
        'MAE': mean_absolute_error(y_reg_test, preds),
        'RMSE': mean_squared_error(y_reg_test, preds) ** 0.5,
        'R2': r2_score(y_reg_test, preds),
    })
    trained_reg[name] = model

reg_df = pd.DataFrame(reg_results).sort_values('R2', ascending=False).reset_index(drop=True)
best_reg_name = reg_df.iloc[0]['Model']
best_reg_model = trained_reg[best_reg_name]
print(f'Best regression model: {best_reg_name}')
reg_df


## 6. Train & Compare Classification Models (Attrition)

In [ ]:
clf_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=200),
    'XGBoost': XGBClassifier(random_state=42, n_estimators=200, use_label_encoder=False, eval_metric='logloss', verbosity=0),
}

clf_results = []
trained_clf = {}
for name, model in clf_models.items():
    model.fit(X_train_proc, y_clf_train)
    preds = model.predict(X_test_proc)
    clf_results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_clf_test, preds),
        'Precision': precision_score(y_clf_test, preds, zero_division=0),
        'Recall': recall_score(y_clf_test, preds, zero_division=0),
        'F1': f1_score(y_clf_test, preds, zero_division=0),
    })
    trained_clf[name] = model

clf_df = pd.DataFrame(clf_results).sort_values('F1', ascending=False).reset_index(drop=True)
best_clf_name = clf_df.iloc[0]['Model']
best_clf_model = trained_clf[best_clf_name]
print(f'Best classification model: {best_clf_name}')
clf_df


## 7. Un-normalize Salary
`MonthlyIncome` in this CSV is already scaled to 0–1 — the original dollar values aren't recoverable from the file alone. This uses the documented range of the original **IBM HR Analytics** dataset (**$1,009–$19,999**), assuming standard min-max scaling was used to create this file.

⚠️ **If you know the actual min/max used to scale your data, edit `ORIGINAL_MIN` / `ORIGINAL_MAX` below** — everything downstream updates automatically.

In [ ]:
ORIGINAL_MIN = 1009   # <-- edit if you know the real original MonthlyIncome min
ORIGINAL_MAX = 19999  # <-- edit if you know the real original MonthlyIncome max

def unnormalize_salary(value):
    return value * (ORIGINAL_MAX - ORIGINAL_MIN) + ORIGINAL_MIN

# Show regression errors in real-dollar terms too, for interpretability
reg_df_dollars = reg_df.copy()
reg_df_dollars['MAE ($)'] = reg_df_dollars['MAE'] * (ORIGINAL_MAX - ORIGINAL_MIN)
reg_df_dollars['RMSE ($)'] = reg_df_dollars['RMSE'] * (ORIGINAL_MAX - ORIGINAL_MIN)
reg_df_dollars


## 8. Save Models
(Optional — lets you reload without retraining later.)

In [ ]:
joblib.dump(preprocessor, 'preprocessor.pkl')
joblib.dump(best_reg_model, 'salary_model.pkl')
joblib.dump(best_clf_model, 'attrition_model.pkl')
print('Saved: preprocessor.pkl, salary_model.pkl, attrition_model.pkl')


## 9. Gradio App
Builds input fields automatically from your dataset's columns — no need to hardcode 30+ fields by hand. Salary output is shown in real dollars using Step 7's un-normalization.

In [ ]:
def make_input(col):
    if col in numeric_cols:
        return gr.Slider(minimum=float(X[col].min()), maximum=float(X[col].max()),
                          value=float(X[col].mean()), label=col)
    else:
        choices = sorted(X[col].dropna().unique().tolist())
        return gr.Dropdown(choices=choices, value=choices[0], label=col)

all_cols = numeric_cols + categorical_cols
inputs = [make_input(c) for c in all_cols]

def predict(*values):
    row = pd.DataFrame([dict(zip(all_cols, values))])
    row = row[X.columns]
    proc = preprocessor.transform(row)
    salary_norm = best_reg_model.predict(proc)[0]
    salary_dollars = unnormalize_salary(salary_norm)
    attr_pred = best_clf_model.predict(proc)[0]
    if hasattr(best_clf_model, 'predict_proba'):
        attr_prob = best_clf_model.predict_proba(proc)[0][1]
        attr_text = f"{'Yes' if attr_pred == 1 else 'No'} (risk score: {attr_prob:.2f})"
    else:
        attr_text = 'Yes' if attr_pred == 1 else 'No'
    return f"${salary_dollars:,.0f}", attr_text

demo = gr.Interface(
    fn=predict,
    inputs=inputs,
    outputs=[gr.Textbox(label='Predicted Monthly Income ($)'), gr.Textbox(label='Attrition Risk')],
    title='Employee Salary & Attrition Predictor',
    description=f'Best regression model: {best_reg_name} | Best classification model: {best_clf_name}',
)
demo.launch(share=True)
